Generate tables for static word embeddings related experiments.

In [49]:
import os
import pandas as pd
import json

RUMODEL_NAMES = [
    "fasttext-ru-torch",
    "geowac_tokens_none_fasttextskipgram_300_5_2020-torch",
    "fasttext-ru-torch_in_batch",
    "geowac_tokens_none_fasttextskipgram_300_5_2020-torch_in_batch"
    ]

def get_chain(obj, *path, default=None):
    for part in path:
        try:
            obj = obj[part]
        except (KeyError, IndexError):
            return default
    return obj

def get_score(model_name, pooling_name, task, task_params):
    root_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    path = os.path.join(root_path, "results", model_name, task, pooling_name , f"{task}.json")
    with open(path, "r") as f:
        data = json.load(f)
    return get_chain(data['scores'], *task_params) if model_name in RUMODEL_NAMES else get_chain(data["test"], *task_params)

def get_data(model_name, pooling_name, task):
    root_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    path = os.path.join(root_path, "results", model_name, task, pooling_name , f"{task}.json")
    with open(path, "r") as f:
        data = json.load(f)
    return data

# example usage
get_data("fasttext-ru-torch", "uniform_pca/whitening", "STS16")

{'dataset_revision': '4d8694f8f0e0100860b497b999b3dbed754a0513',
 'task_name': 'STS16',
 'mteb_version': '1.38.20',
 'scores': {'test': [{'pearson': 0.162787,
    'spearman': 0.222063,
    'cosine_pearson': 0.162787,
    'cosine_spearman': 0.222039,
    'manhattan_pearson': 0.236524,
    'manhattan_spearman': 0.268241,
    'euclidean_pearson': 0.234048,
    'euclidean_spearman': 0.265474,
    'main_score': 0.222039,
    'hf_subset': 'default',
    'languages': ['eng-Latn']}]},
 'evaluation_time': 0.4600214958190918,
 'kg_co2_emissions': None}

In [26]:
get_score("GoogleNews-vectors-negative300-torch", "uniform_zca/whitening", "STS16", ("cos_sim", "spearman"))

0.6104724957778118

In [27]:
def create_pandas_tables(model_names, pooling_names, tasks):
    """
    Creates a dictionary of pandas DataFrames, one per model_name.
    Each DataFrame has:
      - Rows indexed by pooling_names (in the given order).
      - Columns = tasks list + 'Average' column.
    """
    results = {}
    for model_name in model_names:
        # Collect scores in a 2D list where each row corresponds
        # to a specific pooling_name, and each column to a task.
        table_data = []
        for pooling_name in pooling_names:
            row_scores = []
            for task in tasks:
                try:
                    print(f"{model_name}_{pooling_name}_{task}")
                    score = get_score(model_name, pooling_name, task, TASK_MAP[task])
                except FileNotFoundError:
                    score = None
                row_scores.append(score)
            table_data.append(row_scores)

        # Create a DataFrame from the collected data
        df = pd.DataFrame(table_data, index=pooling_names, columns=tasks)
        # Calculate the average score across all tasks
        df["Average"] = df.mean(axis=1)
        # (Optional) name your index for clarity
        df.index.name = "Pooling Name"

        df = df.mul(100).round(2)

        results[model_name] = df

    return results

In [ ]:
# import os
# import shutil
# from pathlib import Path

# def fix_folder_structure():
#     # Base results directory
#     results_dir = Path("../results")
    
#     # Directories to process
#     dirs_to_process = [
#         "geowac_tokens_none_fasttextskipgram_300_5_2020-torch",
#         "fasttext-ru-torch"
#     ]
    
#     for model_dir in dirs_to_process:
#         model_path = results_dir / model_dir
        
#         # Walk through all subdirectories
#         for root, dirs, files in os.walk(model_path):
#             root_path = Path(root)
            
#             # Check if we're in a problematic directory
#             if "no_model_name_available" in root_path.parts:
#                 # Get the parent directory (the actual model directory)
#                 parent_dir = root_path.parent
                
#                 # Get all files in the problematic directory
#                 for file in files:
#                     src_path = root_path / file
#                     dst_path = parent_dir / file
                    
#                     # Move the file up one level
#                     print(f"Moving {src_path} to {dst_path}")
#                     shutil.move(str(src_path), str(dst_path))
                
#                 # Remove the empty directories
#                 try:
#                     os.rmdir(str(root_path))
#                     os.rmdir(str(root_path.parent))
#                 except OSError:
#                     print(f"Could not remove directory {root_path} or its parent")

In [72]:
import os
import shutil

def remove_no_model_folders(root_path):
    """
    Recursively search through a directory and remove any subfolder named 'no_model_name_availabe'
    
    Args:
        root_path (str): The root directory path to start searching from
    """
    # Walk through the directory tree
    for dirpath, dirnames, filenames in os.walk(root_path, topdown=False):
        # Check if 'no_model_name_availabe' is in the current directory's subdirectories
        if 'no_model_name_available' in dirnames:
            folder_to_remove = os.path.join(dirpath, 'no_model_name_available')
            try:
                print(f"Removing folder: {folder_to_remove}")
                shutil.rmtree(folder_to_remove)
            except Exception as e:
                print(f"Error removing folder {folder_to_remove}: {str(e)}")

In [108]:
import os
import shutil

def copy_from_no_revision_to_parent(root_path):
    """
    Recursively search through a directory and copy files from 'no_model_name_available/no_revision_available'
    to their parent directory
    
    Args:
        root_path (str): The root directory path to start searching from
    """
    # Walk through the directory tree
    for dirpath, dirnames, filenames in os.walk(root_path):
        # Check if 'no_model_name_available' is in the current directory's subdirectories
        if 'no_model_name_available' in dirnames:
            no_model_path = os.path.join(dirpath, 'no_model_name_available')
            no_revision_path = os.path.join(no_model_path, 'no_revision_available')
            
            # Check if no_revision_available exists
            if os.path.exists(no_revision_path) and os.path.isdir(no_revision_path):
                try:
                    # Get all files in no_revision_available
                    files_to_copy = [f for f in os.listdir(no_revision_path) 
                                   if os.path.isfile(os.path.join(no_revision_path, f))]
                    
                    # Copy each file to the parent directory
                    for file in files_to_copy:
                        src_path = os.path.join(no_revision_path, file)
                        dst_path = os.path.join(dirpath, file)
                        
                        # If file already exists in parent, add a suffix
                        if os.path.exists(dst_path):
                            base, ext = os.path.splitext(file)
                            counter = 1
                            while os.path.exists(dst_path):
                                new_name = f"{base}_{counter}{ext}"
                                dst_path = os.path.join(dirpath, new_name)
                                counter += 1
                        
                        print(f"Copying {src_path} to {dst_path}")
                        shutil.copy2(src_path, dst_path)
                        
                except Exception as e:
                    print(f"Error processing {no_revision_path}: {str(e)}")

In [115]:
# remove_no_model_folders("../results/fasttext-ru-torch_in_batch")

In [114]:
# copy_from_no_revision_to_parent("../results/fasttext-ru-torch_in_batch")

# Table 8 (enwiki)

In [116]:
model_names = [
    "GoogleNews-vectors-negative300-torch", # word2vec
    "average_word_embeddings_glove.840B.300d", # glove
    "fasttext-en-torch",
    "fasttext-en-subword-torch",
    # "fasttext-ru-torch",
    # "geowac_tokens_none_fasttextskipgram_300_5_2020-torch"
]

pooling_names = [
    "normal/mean",
    "uniform_pca/centering_only",
    "uniform_pca/whitening",
    # "uniform_pca/uniform_centering_then_zipfian_whitening_norm",
    # "uniform_pca/uniform_whitening_then_zipfian_whitening_norm",
    # "uniform_zca/centering_only",
    # "uniform_zca/whitening",
    # "uniform_zca/uniform_centering_then_zipfian_whitening_norm",
    # "uniform_zca/uniform_whitening_then_zipfian_whitening_norm",
    # "uniform_cholesky/centering_only",
    # "uniform_cholesky/whitening",
    # "uniform_cholesky/uniform_centering_then_zipfian_whitening_norm",
    # "uniform_cholesky/uniform_whitening_then_zipfian_whitening_norm",
    "zipfian_pca/centering_only",
    "zipfian_pca/whitening",
    # "zipfian_pca/raw_then_zipfian_whitening_dirction",
    # "zipfian_pca/zipfian_whitening_then_uniform_centering_norm",
    # "zipfian_pca/zipfian_whitening_then_uniform_whitening_norm",
    # "zipfian_zca/centering_only",
    # "zipfian_zca/whitening",
    # "zipfian_zca/raw_then_zipfian_whitening_dirction",
    # "zipfian_zca/zipfian_whitening_then_uniform_centering_norm",
    # "zipfian_zca/zipfian_whitening_then_uniform_whitening_norm",
    # "zipfian_cholesky/centering_only",
    # "zipfian_cholesky/whitening",
    # "zipfian_cholesky/raw_then_zipfian_whitening_dirction",
    # "zipfian_cholesky/zipfian_whitening_then_uniform_centering_norm",
    # "zipfian_cholesky/zipfian_whitening_then_uniform_whitening_norm",
    "abtp/component_removal",
    # "sif/sif_w_component_removal",
]

tasks = [
    "STS12",
    "STS13",
    "STS14",
    "STS15",
    "STS16",
    "SICK-R",
    "STSBenchmark",
    "STS17",
    "STS22",
    "ArxivClusteringS2S",
    "TwitterSemEval2015",
    "MedrxivClusteringS2S",
    "SummEval",
    "AmazonCounterfactualClassification"
]

TASK_MAP = {
    "STS12": ("cos_sim", "spearman"),
    "STS13": ("cos_sim", "spearman"),
    "STS14": ("cos_sim", "spearman"),
    "STS15": ("cos_sim", "spearman"),
    "STS16": ("cos_sim", "spearman"),
    "SICK-R": ("cos_sim", "spearman"),
    "STSBenchmark": ("cos_sim", "spearman"),
    "STS17": ("en-en", "cos_sim", "spearman"),
    "STS22": ("en", "cos_sim", "spearman"),
    "ArxivClusteringS2S": ("main_score",),
    "TwitterSemEval2015": ("cos_sim", "f1"),
    "MedrxivClusteringS2S": ("main_score",),
    "SummEval": ("cos_sim", "spearman"),
    "AmazonCounterfactualClassification": ("en", "main_score")
}

all_results = create_pandas_tables(model_names, pooling_names, tasks)

GoogleNews-vectors-negative300-torch_normal/mean_STS12
GoogleNews-vectors-negative300-torch_normal/mean_STS13
GoogleNews-vectors-negative300-torch_normal/mean_STS14
GoogleNews-vectors-negative300-torch_normal/mean_STS15
GoogleNews-vectors-negative300-torch_normal/mean_STS16
GoogleNews-vectors-negative300-torch_normal/mean_SICK-R
GoogleNews-vectors-negative300-torch_normal/mean_STSBenchmark
GoogleNews-vectors-negative300-torch_normal/mean_STS17
GoogleNews-vectors-negative300-torch_normal/mean_STS22
GoogleNews-vectors-negative300-torch_normal/mean_ArxivClusteringS2S
GoogleNews-vectors-negative300-torch_normal/mean_TwitterSemEval2015
GoogleNews-vectors-negative300-torch_normal/mean_MedrxivClusteringS2S
GoogleNews-vectors-negative300-torch_normal/mean_SummEval
GoogleNews-vectors-negative300-torch_normal/mean_AmazonCounterfactualClassification
GoogleNews-vectors-negative300-torch_uniform_pca/centering_only_STS12
GoogleNews-vectors-negative300-torch_uniform_pca/centering_only_STS13
GoogleNew

In [117]:
# glove
all_results["average_word_embeddings_glove.840B.300d"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,STS17,STS22,ArxivClusteringS2S,TwitterSemEval2015,MedrxivClusteringS2S,SummEval,AmazonCounterfactualClassification,Average
Pooling Name,,,,,,,,,,,,,,,
normal/mean,56.46,50.41,51.13,58.60,49.03,57.01,46.17,64.73,50.53,20.96,45.45,22.15,29.18,65.96,47.70
uniform_pca/centering_only,55.54,46.32,49.67,56.03,46.90,56.44,45.17,62.41,51.54,21.06,44.27,22.16,28.70,65.97,46.58
uniform_pca/whitening,53.31,62.45,57.93,68.68,58.69,57.92,52.21,70.44,54.35,23.25,54.23,22.57,30.94,67.04,52.43
zipfian_pca/centering_only,54.52,69.20,60.87,69.82,62.61,58.01,52.25,72.02,49.86,21.10,54.52,22.17,29.34,65.99,53.02
zipfian_pca/whitening,57.76,72.22,67.04,76.80,71.72,61.80,66.92,82.59,59.64,24.74,54.54,24.46,30.14,67.21,58.40
abtp/component_removal,52.67,67.38,59.40,69.53,60.71,58.56,54.28,71.85,55.73,23.00,54.86,22.89,30.53,66.64,53.43


In [118]:
# word2vec
all_results["GoogleNews-vectors-negative300-torch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,STS17,STS22,ArxivClusteringS2S,TwitterSemEval2015,MedrxivClusteringS2S,SummEval,AmazonCounterfactualClassification,Average
Pooling Name,,,,,,,,,,,,,,,
normal/mean,58.57,68.64,63.65,71.73,61.79,61.77,56.98,74.96,50.69,20.60,51.10,22.46,31.73,68.97,54.54
uniform_pca/centering_only,58.17,67.34,62.19,70.15,59.60,61.39,55.85,72.70,51.97,20.62,50.86,22.46,31.01,68.96,53.80
uniform_pca/whitening,56.53,66.95,62.77,72.42,61.05,62.74,56.03,74.24,49.28,21.24,52.24,21.97,31.42,69.94,54.20
zipfian_pca/centering_only,56.89,69.95,65.08,73.91,65.71,62.18,58.84,78.96,53.10,20.59,52.30,22.46,30.56,68.97,55.68
zipfian_pca/whitening,56.16,70.33,67.20,76.60,70.99,62.52,66.50,81.51,58.65,21.85,52.90,23.42,30.92,69.15,57.77
abtp/component_removal,55.53,69.32,63.13,72.25,60.98,62.02,56.98,74.57,52.41,21.60,51.77,22.39,31.14,70.04,54.58


In [119]:
# fasttext-en-torch
all_results["fasttext-en-torch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,STS17,STS22,ArxivClusteringS2S,TwitterSemEval2015,MedrxivClusteringS2S,SummEval,AmazonCounterfactualClassification,Average
Pooling Name,,,,,,,,,,,,,,,
normal/mean,57.94,68.97,62.37,72.26,63.59,59.99,59.82,71.13,52.04,25.72,56.45,23.57,31.46,71.60,55.49
uniform_pca/centering_only,59.73,55.02,55.16,64.22,53.39,58.85,52.46,67.65,50.29,25.85,51.11,23.57,30.99,71.57,51.42
uniform_pca/whitening,52.46,59.01,53.90,65.33,52.61,58.34,48.60,63.71,49.42,24.74,53.52,22.28,31.07,72.33,50.52
zipfian_pca/centering_only,58.30,71.69,64.57,74.10,67.59,60.75,59.40,75.38,51.50,25.68,56.14,23.57,30.19,71.60,56.46
zipfian_pca/whitening,58.86,73.85,68.43,78.07,74.00,62.85,69.55,80.70,58.34,28.20,56.83,25.11,30.70,73.90,59.96
abtp/component_removal,58.35,69.09,60.82,71.99,60.76,60.34,57.02,71.15,52.77,27.63,54.30,23.81,31.50,72.15,55.12


In [120]:
# fasttext-en-subword-torch
all_results["fasttext-en-subword-torch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,STS17,STS22,ArxivClusteringS2S,TwitterSemEval2015,MedrxivClusteringS2S,SummEval,AmazonCounterfactualClassification,Average
Pooling Name,,,,,,,,,,,,,,,
normal/mean,49.10,47.34,51.94,61.99,51.54,53.60,50.43,58.14,44.88,11.36,53.78,14.21,31.29,70.57,46.44
uniform_pca/centering_only,49.21,43.13,49.89,62.03,49.70,54.56,46.91,59.23,44.11,11.38,52.81,14.21,31.04,70.60,45.63
uniform_pca/whitening,45.12,41.00,47.30,62.08,48.85,54.80,43.55,59.57,45.59,10.79,52.53,13.69,30.19,72.31,44.81
zipfian_pca/centering_only,48.68,55.03,54.07,60.23,58.41,54.64,50.38,66.08,47.42,11.37,55.51,14.20,31.12,70.58,48.41
zipfian_pca/whitening,61.22,60.68,63.18,73.59,69.87,59.82,68.20,78.51,52.03,15.29,57.21,18.28,31.05,73.66,55.90
abtp/component_removal,49.64,41.79,48.81,60.84,47.57,55.09,44.23,58.39,46.53,11.67,51.24,14.37,31.66,71.01,45.21


# Table 9 (test set frequency)

In [121]:
model_names = [
    "GoogleNews-vectors-negative300-torch_in_batch",
    "average_word_embeddings_glove.840B.300d_in_batch", 
    "fasttext-en-torch_in_batch",
    "fasttext-en-subword-torch_in_batch",
]

pooling_names = [
    "normal/mean",
    "uniform_pca/centering_only",
    "uniform_pca/whitening",
    "uniform_zca/centering_only",
    "uniform_zca/whitening",
    "uniform_cholesky/centering_only",
    "uniform_cholesky/whitening",
    "zipfian_pca/centering_only",
    "zipfian_pca/whitening",
    "zipfian_zca/centering_only",
    "zipfian_zca/whitening",
    "zipfian_cholesky/centering_only",
    "zipfian_cholesky/whitening",
    "abtp/component_removal",
#    "sif/sif_w_component_removal",
]

tasks = [
    "STS12",
    "STS13",
    "STS14",
    "STS15",
    "STS16",
    "SICK-R",
    "STSBenchmark",
]

sim = "cos_sim"
cor = "spearman"

all_results = create_pandas_tables(model_names, pooling_names, tasks)

GoogleNews-vectors-negative300-torch_in_batch_normal/mean_STS12
GoogleNews-vectors-negative300-torch_in_batch_normal/mean_STS13
GoogleNews-vectors-negative300-torch_in_batch_normal/mean_STS14
GoogleNews-vectors-negative300-torch_in_batch_normal/mean_STS15
GoogleNews-vectors-negative300-torch_in_batch_normal/mean_STS16
GoogleNews-vectors-negative300-torch_in_batch_normal/mean_SICK-R
GoogleNews-vectors-negative300-torch_in_batch_normal/mean_STSBenchmark
GoogleNews-vectors-negative300-torch_in_batch_uniform_pca/centering_only_STS12
GoogleNews-vectors-negative300-torch_in_batch_uniform_pca/centering_only_STS13
GoogleNews-vectors-negative300-torch_in_batch_uniform_pca/centering_only_STS14
GoogleNews-vectors-negative300-torch_in_batch_uniform_pca/centering_only_STS15
GoogleNews-vectors-negative300-torch_in_batch_uniform_pca/centering_only_STS16
GoogleNews-vectors-negative300-torch_in_batch_uniform_pca/centering_only_SICK-R
GoogleNews-vectors-negative300-torch_in_batch_uniform_pca/centering_o

In [122]:
# glove
all_results["average_word_embeddings_glove.840B.300d_in_batch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,Average
Pooling Name,,,,,,,,
normal/mean,57.71,50.29,50.61,58.38,48.76,56.76,46.22,52.67
uniform_pca/centering_only,56.32,61.17,52.68,64.80,55.80,57.98,47.94,56.67
uniform_pca/whitening,51.67,60.94,57.14,70.09,63.08,55.14,53.16,58.74
uniform_zca/centering_only,56.32,61.17,52.68,64.80,55.80,57.98,47.94,56.67
uniform_zca/whitening,51.67,60.94,57.14,70.09,63.08,55.14,53.16,58.74
uniform_cholesky/centering_only,56.32,61.17,52.68,64.80,55.80,57.98,47.94,56.67
uniform_cholesky/whitening,51.67,60.94,57.14,70.09,63.08,55.14,53.16,58.74
zipfian_pca/centering_only,50.69,70.66,61.59,70.19,68.25,60.03,56.64,62.58
zipfian_pca/whitening,61.63,78.36,69.48,76.83,74.08,60.11,71.60,70.30


In [123]:
# word2vec
all_results["GoogleNews-vectors-negative300-torch_in_batch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,Average
Pooling Name,,,,,,,,
normal/mean,59.00,68.92,63.99,72.51,62.25,61.87,57.15,63.67
uniform_pca/centering_only,57.88,70.34,64.24,74.71,65.57,62.47,58.09,64.76
uniform_pca/whitening,58.45,69.42,65.46,76.43,67.78,62.87,60.85,65.89
uniform_zca/centering_only,57.88,70.34,64.24,74.71,65.57,62.47,58.09,64.76
uniform_zca/whitening,58.45,69.42,65.46,76.43,67.78,62.87,60.85,65.89
uniform_cholesky/centering_only,57.88,70.34,64.24,74.71,65.57,62.47,58.09,64.76
uniform_cholesky/whitening,58.45,69.42,65.46,76.43,67.78,62.87,60.85,65.89
zipfian_pca/centering_only,55.02,71.47,65.81,74.36,69.52,62.92,61.02,65.73
zipfian_pca/whitening,59.37,76.92,69.48,76.42,73.56,60.07,70.42,69.46


In [124]:
# fasttext-en-torch
all_results["fasttext-en-torch_in_batch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,Average
Pooling Name,,,,,,,,
normal/mean,58.23,69.36,62.89,73.09,64.25,60.22,60.27,64.04
uniform_pca/centering_only,60.60,69.51,61.09,73.92,64.49,61.14,57.42,64.02
uniform_pca/whitening,55.56,63.51,57.73,70.68,62.40,57.93,54.65,60.35
uniform_zca/centering_only,60.60,69.51,61.09,73.92,64.49,61.14,57.42,64.02
uniform_zca/whitening,55.56,63.51,57.73,70.68,62.40,57.93,54.65,60.35
uniform_cholesky/centering_only,60.60,69.51,61.09,73.92,64.49,61.14,57.42,64.02
uniform_cholesky/whitening,55.56,63.51,57.73,70.68,62.40,57.93,54.65,60.35
zipfian_pca/centering_only,55.92,73.36,65.72,74.12,72.18,62.30,62.95,66.65
zipfian_pca/whitening,62.20,79.35,71.03,77.95,76.28,60.66,73.56,71.58


In [125]:
# fasttext-en-subword-torch
all_results["fasttext-en-subword-torch_in_batch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,Average
Pooling Name,,,,,,,,
normal/mean,51.37,51.49,54.57,62.75,52.97,53.53,52.41,54.16
uniform_pca/centering_only,51.31,44.80,49.66,62.27,47.43,54.86,43.12,50.49
uniform_pca/whitening,51.52,49.33,53.51,68.28,58.34,56.94,51.69,55.66
uniform_zca/centering_only,51.31,44.80,49.66,62.27,47.43,54.86,43.12,50.49
uniform_zca/whitening,51.52,49.33,53.51,68.28,58.34,56.94,51.69,55.66
uniform_cholesky/centering_only,51.31,44.80,49.66,62.27,47.43,54.86,43.12,50.49
uniform_cholesky/whitening,51.52,49.33,53.51,68.28,58.34,56.94,51.69,55.66
zipfian_pca/centering_only,43.15,53.40,53.67,63.05,59.09,56.57,47.16,53.73
zipfian_pca/whitening,60.87,72.21,67.79,75.86,73.88,60.52,70.99,68.87


# Table 10 (JSTS)

In [84]:
# Wiki frequency
model_names = [
    "fasttext-ja-torch"
]

pooling_names = [
    "normal/mean",
    "uniform_whitening/centering_only",
    "uniform_whitening/whitening",
    "zipfian_whitening/centering_only",
    "zipfian_whitening/whitening",
    "abtp/component_removal",
    "sif/sif_w_component_removal",
]

tasks = [
    "JSTS"
]

sim = "cos_sim"
cor = "spearman"

all_results = create_pandas_tables(model_names, pooling_names, tasks)
all_results["fasttext-ja-torch"]

fasttext-ja-torch_normal/mean_JSTS


KeyError: 'JSTS'

In [85]:
# Test set frequency
model_names = [
    "fasttext-ja-torch_in_batch"
]

pooling_names = [
    "normal/mean",
    "uniform_whitening/centering_only",
    "uniform_whitening/whitening",
    "zipfian_whitening/centering_only",
    "zipfian_whitening/whitening",
    "abtp/component_removal",
]

tasks = [
    "JSTS"
]

sim = "cos_sim"
cor = "spearman"

all_results = create_pandas_tables(model_names, pooling_names, tasks)
all_results["fasttext-ja-torch_in_batch"]

fasttext-ja-torch_in_batch_normal/mean_JSTS


KeyError: 'JSTS'

# Table 11 (norm / direction)

In [126]:
model_names = [
    "GoogleNews-vectors-negative300-torch", # word2vec
    "average_word_embeddings_glove.840B.300d", # glove
    "fasttext-en-torch",
    "fasttext-en-subword-torch",
]

pooling_names = [
    "normal/raw_then_zipfian_whitening_norm",
    "zipfian_whitening/raw_then_zipfian_whitening_dirction",
    "uniform_whitening/uniform_centering_then_zipfian_whitening_norm",
    "uniform_whitening/uniform_whitening_then_zipfian_whitening_norm",
    "zipfian_whitening/zipfian_whitening_then_uniform_centering_norm",
    "zipfian_whitening/zipfian_whitening_then_uniform_whitening_norm",
]

tasks = [
    "STS12",
    "STS13",
    "STS14",
    "STS15",
    "STS16",
    "SICK-R",
    "STSBenchmark",
]

sim = "cos_sim"
cor = "spearman"

all_results = create_pandas_tables(model_names, pooling_names, tasks)

GoogleNews-vectors-negative300-torch_normal/raw_then_zipfian_whitening_norm_STS12
GoogleNews-vectors-negative300-torch_normal/raw_then_zipfian_whitening_norm_STS13
GoogleNews-vectors-negative300-torch_normal/raw_then_zipfian_whitening_norm_STS14
GoogleNews-vectors-negative300-torch_normal/raw_then_zipfian_whitening_norm_STS15
GoogleNews-vectors-negative300-torch_normal/raw_then_zipfian_whitening_norm_STS16
GoogleNews-vectors-negative300-torch_normal/raw_then_zipfian_whitening_norm_SICK-R
GoogleNews-vectors-negative300-torch_normal/raw_then_zipfian_whitening_norm_STSBenchmark
GoogleNews-vectors-negative300-torch_zipfian_whitening/raw_then_zipfian_whitening_dirction_STS12
GoogleNews-vectors-negative300-torch_zipfian_whitening/raw_then_zipfian_whitening_dirction_STS13
GoogleNews-vectors-negative300-torch_zipfian_whitening/raw_then_zipfian_whitening_dirction_STS14
GoogleNews-vectors-negative300-torch_zipfian_whitening/raw_then_zipfian_whitening_dirction_STS15
GoogleNews-vectors-negative300

In [127]:
# glove
# TODO: the current scores in the paper seems to be using pearson correlation, not spearman. check out why.
all_results["average_word_embeddings_glove.840B.300d"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,Average
Pooling Name,,,,,,,,
normal/raw_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/raw_then_zipfian_whitening_dirction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_whitening/uniform_centering_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_whitening/uniform_whitening_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/zipfian_whitening_then_uniform_centering_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/zipfian_whitening_then_uniform_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [128]:
# word2vec
all_results['GoogleNews-vectors-negative300-torch']

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,Average
Pooling Name,,,,,,,,
normal/raw_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/raw_then_zipfian_whitening_dirction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_whitening/uniform_centering_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_whitening/uniform_whitening_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/zipfian_whitening_then_uniform_centering_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/zipfian_whitening_then_uniform_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [129]:
all_results['fasttext-en-torch']

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,Average
Pooling Name,,,,,,,,
normal/raw_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/raw_then_zipfian_whitening_dirction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_whitening/uniform_centering_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_whitening/uniform_whitening_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/zipfian_whitening_then_uniform_centering_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/zipfian_whitening_then_uniform_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
all_results['fasttext-en-subword-torch']

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,Average
Pooling Name,,,,,,,,
normal/raw_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/raw_then_zipfian_whitening_dirction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_whitening/uniform_centering_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_whitening/uniform_whitening_then_zipfian_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/zipfian_whitening_then_uniform_centering_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_whitening/zipfian_whitening_then_uniform_whitening_norm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Russian models (eng_tasks)

In [134]:
model_names = [
    "fasttext-ru-torch",
    "geowac_tokens_none_fasttextskipgram_300_5_2020-torch",
    "fasttext-ru-torch_in_batch",
    "geowac_tokens_none_fasttextskipgram_300_5_2020-torch_in_batch"
]

pooling_names = [
    "normal/mean",
    "uniform_pca/centering_only",
    "uniform_pca/whitening",
    # "uniform_pca/uniform_centering_then_zipfian_whitening_norm",
    # "uniform_pca/uniform_whitening_then_zipfian_whitening_norm",
    "zipfian_pca/centering_only",
    "zipfian_pca/whitening",
    # "zipfian_pca/raw_then_zipfian_whitening_dirction",
    # "zipfian_pca/zipfian_whitening_then_uniform_centering_norm",
    # "zipfian_pca/zipfian_whitening_then_uniform_whitening_norm",
    "abtp/component_removal",
    "sif/sif_w_component_removal",
]

tasks = [
    "STS12",
    "STS13",
    "STS14",
    "STS15",
    "STS16",
    "SICK-R",
    "STSBenchmark",
    "STS17",
    "STS22",

]

TASK_MAP = {
    "STS12": ("test", 0, "cosine_spearman",),
    "STS13": ("test", 0,"cosine_spearman",),
    "STS14": ("test", 0,"cosine_spearman",),
    "STS15": ("test", 0,"cosine_spearman",),
    "STS16": ("test", 0,"cosine_spearman",),
    "SICK-R": ("test", 0,"cosine_spearman"),
    "STSBenchmark": ("test", 0,"cosine_spearman",),
    "STS17": ("test", 4, "cosine_spearman"),
    "STS22": ("test", 0, "cosine_spearman"),
}

all_results = create_pandas_tables(model_names, pooling_names, tasks)

fasttext-ru-torch_normal/mean_STS12
fasttext-ru-torch_normal/mean_STS13
fasttext-ru-torch_normal/mean_STS14
fasttext-ru-torch_normal/mean_STS15
fasttext-ru-torch_normal/mean_STS16
fasttext-ru-torch_normal/mean_SICK-R
fasttext-ru-torch_normal/mean_STSBenchmark
fasttext-ru-torch_normal/mean_STS17
fasttext-ru-torch_normal/mean_STS22
fasttext-ru-torch_uniform_pca/centering_only_STS12
fasttext-ru-torch_uniform_pca/centering_only_STS13
fasttext-ru-torch_uniform_pca/centering_only_STS14
fasttext-ru-torch_uniform_pca/centering_only_STS15
fasttext-ru-torch_uniform_pca/centering_only_STS16
fasttext-ru-torch_uniform_pca/centering_only_SICK-R
fasttext-ru-torch_uniform_pca/centering_only_STSBenchmark
fasttext-ru-torch_uniform_pca/centering_only_STS17
fasttext-ru-torch_uniform_pca/centering_only_STS22
fasttext-ru-torch_uniform_pca/whitening_STS12
fasttext-ru-torch_uniform_pca/whitening_STS13
fasttext-ru-torch_uniform_pca/whitening_STS14
fasttext-ru-torch_uniform_pca/whitening_STS15
fasttext-ru-torch

KeyError: 'test'

In [131]:
all_results["fasttext-ru-torch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,STS17,STS22,Average
Pooling Name,,,,,,,,,,
normal/mean,17.53,11.94,23.92,33.22,23.30,40.48,12.78,36.63,36.22,26.23
uniform_pca/centering_only,17.21,11.74,23.68,33.14,23.19,40.42,12.38,36.68,36.25,26.08
uniform_pca/whitening,17.17,10.62,22.24,34.98,22.20,40.18,10.63,37.68,36.30,25.78
zipfian_pca/centering_only,17.57,13.74,25.27,33.96,24.38,40.85,14.19,37.19,35.72,26.98
zipfian_pca/whitening,22.32,20.46,31.02,45.32,30.25,44.78,22.04,49.63,41.71,34.17
abtp/component_removal,20.30,10.70,22.97,35.13,22.27,41.56,12.71,36.73,36.72,26.57
sif/sif_w_component_removal,34.10,28.56,31.98,42.56,29.33,45.19,27.16,NaN,NaN,34.13


In [132]:
all_results["geowac_tokens_none_fasttextskipgram_300_5_2020-torch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,STS17,STS22,Average
Pooling Name,,,,,,,,,,
normal/mean,49.48,28.98,40.54,51.27,36.45,52.24,42.16,50.27,42.48,43.77
uniform_pca/centering_only,48.95,29.25,40.55,51.08,36.40,52.25,41.50,51.17,42.33,43.72
uniform_pca/whitening,50.50,29.22,41.24,52.45,36.38,54.18,39.86,54.23,41.01,44.34
zipfian_pca/centering_only,48.62,29.49,40.76,51.24,36.74,52.19,41.80,51.17,42.07,43.79
zipfian_pca/whitening,51.23,31.76,42.95,55.10,39.40,53.79,44.46,55.38,43.13,46.36
abtp/component_removal,51.48,27.45,39.86,49.71,35.20,53.77,39.41,51.08,42.37,43.37
sif/sif_w_component_removal,45.06,30.58,42.81,53.76,38.47,53.20,46.24,NaN,NaN,44.30


In [133]:
all_results["fasttext-ru-torch_in_batch"]

KeyError: 'fasttext-ru-torch_in_batch'

In [ ]:
all_results["geowac_tokens_none_fasttextskipgram_300_5_2020-torch"]

,STS12,STS13,STS14,STS15,STS16,SICK-R,STSBenchmark,STS17,STS22,Average
Pooling Name,,,,,,,,,,
normal/mean,49.48,28.98,40.54,51.27,36.45,52.24,42.16,50.27,42.48,43.77
uniform_pca/centering_only,48.95,29.25,40.55,51.08,36.40,52.25,41.50,51.17,42.33,43.72
uniform_pca/whitening,50.50,29.22,41.24,52.45,36.38,54.18,39.86,54.23,41.01,44.34
zipfian_pca/centering_only,48.62,29.49,40.76,51.24,36.74,52.19,41.80,51.17,42.07,43.79
zipfian_pca/whitening,51.23,31.76,42.95,55.10,39.40,53.79,44.46,55.38,43.13,46.36
abtp/component_removal,51.48,27.45,39.86,49.71,35.20,53.77,39.41,51.08,42.37,43.37
sif/sif_w_component_removal,45.06,30.58,42.81,53.76,38.47,53.20,46.24,NaN,NaN,44.30


### Russian models (russian tasks)

In [94]:
model_names = [
    "fasttext-ru-torch",
    "geowac_tokens_none_fasttextskipgram_300_5_2020-torch",
    "fasttext-ru-torch_in_batch",
    "geowac_tokens_none_fasttextskipgram_300_5_2020-torch_in_batch"
]

pooling_names = [
    "normal/mean",
    "uniform_pca/centering_only",
    "uniform_pca/whitening",
    # "uniform_pca/uniform_centering_then_zipfian_whitening_norm",
    # "uniform_pca/uniform_whitening_then_zipfian_whitening_norm",
    "zipfian_pca/centering_only",
    "zipfian_pca/whitening",
    # "zipfian_pca/raw_then_zipfian_whitening_dirction",
    # "zipfian_pca/zipfian_whitening_then_uniform_centering_norm",
    # "zipfian_pca/zipfian_whitening_then_uniform_whitening_norm",
    "abtp/component_removal",
]

tasks = [
    "STS22",
    "GeoreviewClassification",
    "GeoreviewClusteringP2P",
    "HeadlineClassification",
    "InappropriatenessClassification",
    "KinopoiskClassification",
    "RiaNewsRetrieval",
    "RuBQRetrieval",
    "RuReviewsClassification",
    "RuSciBenchGRNTIClassification",
    "RuSciBenchGRNTIClusteringP2P",
    "RuSciBenchOECDClassification",
    "RuSciBenchOECDClusteringP2P",
    "RuSTSBenchmarkSTS",
    "TERRa",
    "RuBQReranking",
    "CEDRClassification",
    "SensitiveTopicsClassification"
]

TASK_MAP = {
    "STS22": ("test", 6, "cosine_spearman"),
    "GeoreviewClassification": ("test", 0, "f1"),
    "GeoreviewClusteringP2P": ("test", 0, "main_score"),
    "HeadlineClassification": ("test", 0, "f1"),
    "InappropriatenessClassification": ("test", 0, "f1"),
    "KinopoiskClassification": ("test", 0, "f1"),
    "RiaNewsRetrieval": ("test", 0, "main_score"),
    "RuBQRetrieval": ("test", 0, "main_score"),
    "RuReviewsClassification": ("test", 0, "f1"),
    "RuSciBenchGRNTIClassification": ("test", 0, "f1"),
    "RuSciBenchGRNTIClusteringP2P": ("test", 0, "main_score"),
    "RuSciBenchOECDClassification": ("test", 0, "f1"),
    "RuSciBenchOECDClusteringP2P": ("test", 0, "main_score"),
    "RuSTSBenchmarkSTS": ("test", 0, "cosine_spearman"),
    "TERRa": ("dev", 0, "cosine_f1"),
    "RuBQReranking": ("test", 0, "main_score"),
    "CEDRClassification": ("test", 0, "f1"),
    "SensitiveTopicsClassification": ("test", 0, "f1")
}

all_results = create_pandas_tables(model_names, pooling_names, tasks)

fasttext-ru-torch_normal/mean_STS22
fasttext-ru-torch_normal/mean_GeoreviewClassification
fasttext-ru-torch_normal/mean_GeoreviewClusteringP2P
fasttext-ru-torch_normal/mean_HeadlineClassification
fasttext-ru-torch_normal/mean_InappropriatenessClassification
fasttext-ru-torch_normal/mean_KinopoiskClassification
fasttext-ru-torch_normal/mean_RiaNewsRetrieval
fasttext-ru-torch_normal/mean_RuBQRetrieval
fasttext-ru-torch_normal/mean_RuReviewsClassification
fasttext-ru-torch_normal/mean_RuSciBenchGRNTIClassification
fasttext-ru-torch_normal/mean_RuSciBenchGRNTIClusteringP2P
fasttext-ru-torch_normal/mean_RuSciBenchOECDClassification
fasttext-ru-torch_normal/mean_RuSciBenchOECDClusteringP2P
fasttext-ru-torch_normal/mean_RuSTSBenchmarkSTS
fasttext-ru-torch_normal/mean_TERRa
fasttext-ru-torch_normal/mean_RuBQReranking
fasttext-ru-torch_normal/mean_CEDRClassification
fasttext-ru-torch_normal/mean_SensitiveTopicsClassification
fasttext-ru-torch_uniform_pca/centering_only_STS22
fasttext-ru-torch_u

In [95]:
all_results["fasttext-ru-torch"]

,STS22,GeoreviewClassification,GeoreviewClusteringP2P,HeadlineClassification,InappropriatenessClassification,KinopoiskClassification,RiaNewsRetrieval,RuBQRetrieval,RuReviewsClassification,RuSciBenchGRNTIClassification,RuSciBenchGRNTIClusteringP2P,RuSciBenchOECDClassification,RuSciBenchOECDClusteringP2P,RuSTSBenchmarkSTS,TERRa,RuBQReranking,CEDRClassification,SensitiveTopicsClassification,Average
Pooling Name,,,,,,,,,,,,,,,,,,,
normal/mean,1.24,9.92,4.48,12.00,34.30,26.69,0.43,0.92,22.80,3.63,11.17,3.42,9.58,-3.25,66.23,8.18,3.69,0.12,11.97
uniform_pca/centering_only,-0.77,9.91,4.75,12.22,34.30,26.44,0.43,0.95,22.79,3.63,11.56,3.46,9.64,-3.27,66.23,8.18,3.74,0.12,11.91
uniform_pca/whitening,-1.63,9.77,4.60,12.30,33.93,26.38,0.42,1.09,22.61,3.78,11.81,3.46,10.29,-3.30,66.67,8.26,3.94,0.00,11.91
zipfian_pca/centering_only,1.13,9.91,4.50,11.91,34.30,26.81,0.45,0.86,22.80,3.59,11.45,3.36,9.82,-3.25,66.37,8.15,4.04,0.11,12.02
zipfian_pca/whitening,0.52,9.87,4.83,12.89,34.01,26.38,0.42,1.10,22.90,3.72,11.71,3.43,10.25,-3.28,66.38,8.33,3.94,0.00,12.08
abtp/component_removal,-1.94,9.57,4.77,12.34,33.80,25.36,0.42,0.92,22.33,3.75,11.87,3.47,10.45,-3.25,66.52,8.23,3.95,0.00,11.81


In [96]:
all_results["geowac_tokens_none_fasttextskipgram_300_5_2020-torch"]

,STS22,GeoreviewClassification,GeoreviewClusteringP2P,HeadlineClassification,InappropriatenessClassification,KinopoiskClassification,RiaNewsRetrieval,RuBQRetrieval,RuReviewsClassification,RuSciBenchGRNTIClassification,RuSciBenchGRNTIClusteringP2P,RuSciBenchOECDClassification,RuSciBenchOECDClusteringP2P,RuSTSBenchmarkSTS,TERRa,RuBQReranking,CEDRClassification,SensitiveTopicsClassification,Average
Pooling Name,,,,,,,,,,,,,,,,,,,
normal/mean,1.83,9.54,3.15,13.43,34.31,26.43,0.56,1.27,22.33,3.85,11.48,3.43,9.56,-2.71,66.37,8.30,4.13,0.03,12.07
uniform_pca/centering_only,0.17,9.39,2.97,13.38,34.31,26.30,0.57,1.27,22.18,3.84,11.74,3.48,9.70,-2.73,66.38,8.30,4.13,0.03,11.97
uniform_pca/whitening,0.28,9.36,2.81,13.13,34.13,26.41,0.58,1.25,22.58,3.89,11.29,3.54,9.62,-2.59,66.52,8.19,3.96,0.00,11.94
zipfian_pca/centering_only,0.59,9.30,3.18,13.51,34.31,26.30,0.56,1.27,22.00,3.81,11.95,3.40,10.05,-2.70,66.23,8.24,4.16,0.03,12.01
zipfian_pca/whitening,-1.15,9.30,3.32,13.50,34.25,26.08,0.54,1.26,22.75,3.96,11.70,3.57,9.88,-2.60,66.23,8.27,3.97,0.00,11.93
abtp/component_removal,-0.17,9.19,2.90,13.38,34.02,26.40,0.55,1.31,22.25,3.93,11.89,3.59,10.15,-2.63,66.67,8.24,3.97,0.00,11.98


In [97]:
all_results["fasttext-ru-torch_in_batch"]

,STS22,GeoreviewClassification,GeoreviewClusteringP2P,HeadlineClassification,InappropriatenessClassification,KinopoiskClassification,RiaNewsRetrieval,RuBQRetrieval,RuReviewsClassification,RuSciBenchGRNTIClassification,RuSciBenchGRNTIClusteringP2P,RuSciBenchOECDClassification,RuSciBenchOECDClusteringP2P,RuSTSBenchmarkSTS,TERRa,RuBQReranking,CEDRClassification,SensitiveTopicsClassification,Average
Pooling Name,,,,,,,,,,,,,,,,,,,
normal/mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_pca/centering_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_pca/whitening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_pca/centering_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_pca/whitening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
abtp/component_removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [98]:
all_results["geowac_tokens_none_fasttextskipgram_300_5_2020-torch_in_batch"]

,STS22,GeoreviewClassification,GeoreviewClusteringP2P,HeadlineClassification,InappropriatenessClassification,KinopoiskClassification,RiaNewsRetrieval,RuBQRetrieval,RuReviewsClassification,RuSciBenchGRNTIClassification,RuSciBenchGRNTIClusteringP2P,RuSciBenchOECDClassification,RuSciBenchOECDClusteringP2P,RuSTSBenchmarkSTS,TERRa,RuBQReranking,CEDRClassification,SensitiveTopicsClassification,Average
Pooling Name,,,,,,,,,,,,,,,,,,,
normal/mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_pca/centering_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
uniform_pca/whitening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_pca/centering_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zipfian_pca/whitening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
abtp/component_removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
